In [5]:
!pip install imbalanced-learn


Defaulting to user installation because normal site-packages is not writeable
  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached sklearn_compat-0.1.5-py3-none-any.whl.metadata (20 kB)
Using cached imbalanced_learn-0.14.1-py3-none-any.whl (235 kB)
Using cached sklearn_compat-0.1.5-py3-none-any.whl (20 kB)

   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]




[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import sys
!{sys.executable} -m pip install imbalanced-learn


  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached sklearn_compat-0.1.5-py3-none-any.whl.metadata (20 kB)
Using cached imbalanced_learn-0.14.1-py3-none-any.whl (235 kB)
Using cached sklearn_compat-0.1.5-py3-none-any.whl (20 kB)

   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]



In [1]:
from imblearn.over_sampling import SMOTE
print("SMOTE imported successfully!")


SMOTE imported successfully!


In [5]:
# ==============================
# 1. IMPORT
# ==============================
import pandas as pd
import numpy as np
import ast

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ==============================
# 2. LOAD DATA
# ==============================
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

# ==============================
# 3. MERGE
# ==============================
df = movies.merge(credits, on="title")

# ==============================
# 4. SELECT COLUMNS
# ==============================
df = df[['title','genres','budget','production_companies',
         'production_countries','release_date',
         'cast','crew','vote_average']]

df.rename(columns={'vote_average':'rating'}, inplace=True)

# ==============================
# 5. CLEAN FUNCTIONS
# ==============================
def convert(text):
    return [i['name'] for i in ast.literal_eval(text)]

def get_first(text):
    L = [i['name'] for i in ast.literal_eval(text)]
    return L[0] if len(L)>0 else "unknown"

def get_actor(text):
    L = [i['name'] for i in ast.literal_eval(text)]
    return L[0] if len(L)>0 else "unknown"

def get_director(text):
    for i in ast.literal_eval(text):
        if i['job']=='Director':
            return i['name']
    return "unknown"

# ==============================
# 6. APPLY CLEANING
# ==============================
df['genre_list'] = df['genres'].apply(convert)
df['genre'] = df['genre_list'].apply(lambda x: ",".join(x))

df['studio'] = df['production_companies'].apply(get_first)
df['country'] = df['production_countries'].apply(get_first)

df['actor'] = df['cast'].apply(get_actor)
df['director'] = df['crew'].apply(get_director)

# ==============================
# 7. FILTER ANIMATION
# ==============================
df = df[df['genre'].str.contains("Animation", na=False)]

# ==============================
# 8. CLEAN DATA
# ==============================
df.dropna(inplace=True)
df = df[df['budget'] > 0]
df = df[df['rating'] > 0]

# budget → million
df['budget'] = df['budget'] / 1000000

# ==============================
# 9. SEASON FEATURE
# ==============================
df['release_date'] = pd.to_datetime(df['release_date'])
df['month'] = df['release_date'].dt.month

def get_season(m):
    if m in [5,6,7]:
        return "Summer"
    elif m in [11,12]:
        return "Holiday"
    else:
        return "Normal"

df['season'] = df['month'].apply(get_season)

# ==============================
# 10. SCREEN CATEGORY
# ==============================
def screen_category(b):
    if b >= 100:
        return "Wide"
    elif b >= 40:
        return "Medium"
    else:
        return "Limited"

df['screen'] = df['budget'].apply(screen_category)

# ==============================
# 11. DIRECTOR SCORE
# ==============================
director_score = df.groupby('director')['rating'].mean()
df['director_score'] = df['director'].map(director_score)

# ==============================
# 12. TARGET LABEL
# ==============================
def classify(r):
    if r >= 7:
        return "Hit"
    elif r >= 5:
        return "Average"
    else:
        return "Flop"

df['status'] = df['rating'].apply(classify)

# ==============================
# 13. ENCODING
# ==============================
mlb = MultiLabelBinarizer()
genre_encoded = mlb.fit_transform(df['genre_list'])

studio_encoded = pd.get_dummies(df['studio']).values
country_encoded = pd.get_dummies(df['country']).values
season_encoded = pd.get_dummies(df['season']).values
screen_encoded = pd.get_dummies(df['screen']).values

# actor (top 20 only)
top_actors = df['actor'].value_counts().head(20).index
df['actor'] = df['actor'].apply(lambda x: x if x in top_actors else "other")
actor_encoded = pd.get_dummies(df['actor']).values

# ==============================
# 14. FINAL FEATURES
# ==============================
X = np.hstack((
    genre_encoded,
    studio_encoded,
    country_encoded,
    season_encoded,
    screen_encoded,
    actor_encoded,
    df[['director_score']].values,
    df[['budget']].values
))

y = df['status']

# ==============================
# 15. TRAIN TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# ==============================
# 16. GRIDSEARCH (RF)
# ==============================
param_grid = {
    'n_estimators': [100,150],
    'max_depth': [10,15,None]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=3)
grid.fit(X_train, y_train)

best_rf = grid.best_estimator_

print("Best Params:", grid.best_params_)

# ==============================
# 17. ENSEMBLE MODEL
# ==============================
gb = GradientBoostingClassifier()

model = VotingClassifier(
    estimators=[('rf', best_rf), ('gb', gb)],
    voting='hard'
)

model.fit(X_train, y_train)

# ==============================
# 18. CROSS VALIDATION
# ==============================
cv_scores = cross_val_score(model, X, y, cv=5)
print("CV Score:", cv_scores.mean())

# ==============================
# 19. PREDICTION
# ==============================
y_pred = model.predict(X_test)

# ==============================
# 20. EVALUATION
# ==============================
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ==============================
# 21. SAVE DATASET
# ==============================
df.to_csv("animation_msc_final.csv", index=False)

print("✅ FINAL DATASET & MODEL READY")

Best Params: {'max_depth': 10, 'n_estimators': 150}
CV Score: 0.8131242740998837
Accuracy: 0.8333333333333334
Precision: 0.7755102040816326
Recall: 0.8333333333333334
F1 Score: 0.7980769230769231
Confusion Matrix:
 [[29  0  1]
 [ 3  0  0]
 [ 3  0  6]]
✅ FINAL DATASET & MODEL READY


C:\Users\shiva\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
